In [1]:
# --- PROJECT CONFIG --- #
CONFIG = {
    # BBOX in WGS84: [minx, miny, maxx, maxy]
    # Tiny central Amsterdam tile:
    "bbox": [4.886, 52.370, 4.898, 52.378],

    # Storm & green roof behavior
    "storm_mm": 50.0,  # storm depth (mm)
    "C_roof": 0.9,     # bare roof runoff coefficient
    "Cg": 0.2,         # green roof runoff coefficient after retention is filled
    "R_mm": 12.0,      # green roof retention per storm (mm captured before runoff)

    # Scenario coverage fractions (by roof area)
    "scenarios": [0.10, 0.20, 0.30, 0.40, 0.50],

    # Map outputs
    "save_basemap": True,
    "map_folder": "outputs/maps",
}
CONFIG


{'bbox': [4.886, 52.37, 4.898, 52.378],
 'storm_mm': 50.0,
 'C_roof': 0.9,
 'Cg': 0.2,
 'R_mm': 12.0,
 'scenarios': [0.1, 0.2, 0.3, 0.4, 0.5],
 'save_basemap': True,
 'map_folder': 'outputs/maps'}

In [2]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import osmnx as ox
from shapely.geometry import box

# Make output folders
os.makedirs("outputs", exist_ok=True)
os.makedirs(CONFIG["map_folder"], exist_ok=True)


In [3]:
def fetch_buildings_by_bbox(bbox):
    """
    Fetch OSM building polygons inside bbox and compute area_m2.
    Uses features_from_polygon.
    """
    minx, miny, maxx, maxy = bbox
    poly = box(minx, miny, maxx, maxy)

    # Query OSM: buildings only
    gdf = ox.features_from_polygon(poly, tags={"building": True})

    # Keep only polygonal geometries (roofs are polygons)
    gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    if gdf.empty:
        raise RuntimeError("No OSM building polygons found. Enlarge the bbox or try another area.")

    # Project to a metric CRS for correct area (m²)
    # Try osmnx helper; if it fails, estimate UTM
    try:
        gdf_proj = ox.project_gdf(gdf)
    except Exception:
        gdf_proj = gdf.to_crs(gdf.estimate_utm_crs())

    gdf_proj["area_m2"] = gdf_proj.geometry.area
    return gdf_proj[["geometry", "area_m2"]].reset_index(drop=True)

# Run the fetch
bldgs = fetch_buildings_by_bbox(CONFIG["bbox"])
total_roof_m2 = float(bldgs["area_m2"].sum())
len(bldgs), f"Total roof area: {total_roof_m2:,.0f} m²"


(2225, 'Total roof area: 405,157 m²')

In [4]:
def select_green_roofs(gdf_bldgs: gpd.GeoDataFrame, frac: float) -> gpd.GeoDataFrame:
    """
    Select the largest roofs until their cumulative area reaches 'frac' of total roof area.
    Ensures at least 1 building if frac is extremely small.
    """
    g = gdf_bldgs.sort_values("area_m2", ascending=False).copy()
    g["cum_area"] = g["area_m2"].cumsum()
    total = float(g["area_m2"].sum())
    target = frac * total
    sel = g[g["cum_area"] <= target].drop(columns=["cum_area"])
    if sel.empty and len(g):  # pick the biggest one if target is too small
        sel = g.head(1).drop(columns=["cum_area"])
    return sel


In [5]:
def runoff_roof_m3_with_green(P_mm: float,
                              A_treated_m2: float,
                              A_untreated_m2: float,
                              C_roof: float = 0.9,
                              Cg: float = 0.2,
                              R_mm: float = 12.0) -> float:
    """
    Compute total runoff volume (m³) from roofs under a storm of P_mm:
      - Greened area retains first R_mm (no runoff), then runs off at Cg.
      - Untreated area runs off at C_roof for the whole storm.
    """
    depth_m = P_mm / 1000.0
    excess_m = max(0.0, (P_mm - R_mm)) / 1000.0  # only after retention fills
    runoff_treated   = excess_m * Cg    * A_treated_m2
    runoff_untreated = depth_m * C_roof * A_untreated_m2
    return runoff_treated + runoff_untreated


In [6]:
P      = float(CONFIG["storm_mm"])
C_roof = float(CONFIG["C_roof"])
Cg     = float(CONFIG["Cg"])
R_mm   = float(CONFIG["R_mm"])

base_runoff_m3 = (P / 1000.0) * C_roof * total_roof_m2  # baseline: all bare roof

rows = []
for frac in CONFIG["scenarios"]:
    sel = select_green_roofs(bldgs, frac)
    A_t = float(sel["area_m2"].sum())            # treated roof area
    A_u = total_roof_m2 - A_t                    # untreated roof area
    scen_runoff_m3 = runoff_roof_m3_with_green(P, A_t, A_u, C_roof=C_roof, Cg=Cg, R_mm=R_mm)
    reduction = (base_runoff_m3 - scen_runoff_m3) / base_runoff_m3 if base_runoff_m3 > 0 else 0.0
    rows.append({
        "coverage_frac": frac,
        "treated_roof_m2": A_t,
        "runoff_baseline_m3": base_runoff_m3,
        "runoff_scenario_m3": scen_runoff_m3,
        "reduction_pct": 100.0 * reduction,
        "R_mm": R_mm,
        "Cg": Cg,
    })

results = pd.DataFrame(rows).sort_values("coverage_frac")
results.to_csv("outputs/scenarios.csv", index=False)
results


,coverage_frac,treated_roof_m2,runoff_baseline_m3,runoff_scenario_m3,reduction_pct,R_mm,Cg
0,0.1,38738.619767,18232.080265,16783.255886,7.946566,12.0,0.2
1,0.2,79448.922986,18232.080265,15260.690546,16.297590,12.0,0.2
2,0.3,121363.337505,18232.080265,13693.091443,24.895617,12.0,0.2
3,0.4,162037.880872,18232.080265,12171.863521,33.239305,12.0,0.2
4,0.5,202523.999352,18232.080265,10657.682690,41.544341,12.0,0.2


In [7]:
def save_map_for_frac(bldgs, frac, folder):
    """Plot all buildings (light) and selected roofs (outline) and save PNG."""
    sel = select_green_roofs(bldgs, frac)

    fig, ax = plt.subplots(figsize=(7, 7))
    bldgs.plot(ax=ax, alpha=0.25, linewidth=0.2, edgecolor="k")
    if not sel.empty:
        sel.boundary.plot(ax=ax, linewidth=1.0)

    # lock extent so all images are comparable
    minx, miny, maxx, maxy = bldgs.total_bounds
    ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)

    ax.set_axis_off()
    ax.set_title(f"Green roofs (outline) — {frac*100:.0f}% coverage")

    out_path = os.path.join(folder, f"green_roof_{int(frac*100)}.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.close(fig)
    return out_path, len(sel), float(sel["area_m2"].sum())

for f in CONFIG["scenarios"]:
    path, nsel, area = save_map_for_frac(bldgs, f, CONFIG["map_folder"])
    print(f"Saved: {path} | selected: {nsel} | area: {area:,.0f} m²")


Saved: outputs/maps/green_roof_10.png | selected: 8 | area: 38,739 m²
Saved: outputs/maps/green_roof_20.png | selected: 25 | area: 79,449 m²
Saved: outputs/maps/green_roof_30.png | selected: 58 | area: 121,363 m²
Saved: outputs/maps/green_roof_40.png | selected: 111 | area: 162,038 m²
Saved: outputs/maps/green_roof_50.png | selected: 198 | area: 202,524 m²


In [9]:
import os
import contextily as cx
import matplotlib.pyplot as plt

bm_folder = "outputs/maps_basemap"
os.makedirs(bm_folder, exist_ok=True)

# Good options:
#   cx.providers.OpenStreetMap.Mapnik
#   cx.providers.CartoDB.Positron
#   cx.providers.CartoDB.Voyager
BASEMAP_PROVIDER = cx.providers.CartoDB.Positron  

def save_map_for_frac_basemap(bldgs, frac, folder, provider=BASEMAP_PROVIDER):
    sel = select_green_roofs(bldgs, frac)

    # reproject to Web Mercator (required for slippy-map tiles)
    b3857 = bldgs.to_crs(3857)
    s3857 = sel.to_crs(3857) if not sel.empty else sel

    fig, ax = plt.subplots(figsize=(7, 7))
    b3857.plot(ax=ax, alpha=0.25, linewidth=0.2, edgecolor="k")   # all buildings
    if not s3857.empty:
        s3857.boundary.plot(ax=ax, linewidth=1.1)                 # selected roofs

    # IMPORTANT: pass crs=3857 so contextily knows your axes CRS
    cx.add_basemap(ax, source=provider, crs=3857)
    ax.set_axis_off()
    ax.set_title(f"Green roofs on basemap — {frac*100:.0f}% coverage")

    out_path = os.path.join(folder, f"green_roof_{int(frac*100)}_bm.png")
    plt.tight_layout(); plt.savefig(out_path, dpi=250); plt.close(fig)
    return out_path

# loop all scenarios
for f in CONFIG["scenarios"]:
    print("Saved:", save_map_for_frac_basemap(bldgs, f, bm_folder))


Saved: outputs/maps_basemap/green_roof_10_bm.png
Saved: outputs/maps_basemap/green_roof_20_bm.png
Saved: outputs/maps_basemap/green_roof_30_bm.png
Saved: outputs/maps_basemap/green_roof_40_bm.png
Saved: outputs/maps_basemap/green_roof_50_bm.png
